# ML Zoomcamp 2026 — Homework 3

Official: `materials/homework.md` (do not edit).
Data: `data/course_lead_scoring_2026.csv` (2026 pinned lead-scoring release).
Write-up: `ML_03_HW.md`.

Kernel: **ML Zoomcamp** / this folder `.venv`. Do not use `!pip` / `!wget`.


## 0) Imports


In [8]:
import os
# Import the stdlib os module to set process environment variables.

os.environ.setdefault(r"MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
# Put Matplotlib config/cache on E: to avoid permission issues.

os.environ.setdefault(r"XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
# Use the shared XDG cache directory on E:.

os.environ.setdefault(r"TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
# Pin TEMP to a known folder so temp files stay on E:.

os.environ.setdefault(r"TMP", r"E:\IT_SPACES\AI\.cache\tmp")
# Pin TMP the same way.

os.environ.setdefault("OMP_NUM_THREADS", "1")
# Limit OpenMP threads to 1 to avoid CPU oversubscription.

os.environ.setdefault("MKL_NUM_THREADS", "1")
# Limit Intel MKL threads to 1 for stable, reproducible runs.

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
# Limit OpenBLAS threads to 1.

os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
# Limit NumExpr threads to 1.

import numpy as np
# Numerical arrays and linear algebra.

import pandas as pd
# Tabular data loading and wrangling.

from sklearn.model_selection import train_test_split
# Split the frame into train / validation / test.

from sklearn.metrics import mutual_info_score, accuracy_score
# Mutual information for Q3 and classification accuracy for Q4-Q6.

from sklearn.linear_model import LogisticRegression
# Logistic regression for the classification questions.


## 1) Load data


In [2]:
df = pd.read_csv("data/course_lead_scoring_2026.csv")
# Load the pinned 2026 lead-scoring CSV into a DataFrame.

df.head()
# Preview the first rows and column names.


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

If missing values are present: categorical → `'NA'`, numerical → `0.0`.


In [3]:
df.isnull().sum()
# Count missing values in each column.


lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [ ]:
# Fill missing values here (categorical 'NA', numerical 0.0).
categorical_cols = df.select_dtypes(include=['object', 'string']).columns
# Keep only object/string column names.

df[categorical_cols] = df[categorical_cols].fillna('NA')
# Fill categorical NaNs with 'NA' so they become a real category.

numerical_cols = df.select_dtypes(include=['number']).columns
# Keep only numeric column names.

df[numerical_cols] = df[numerical_cols].fillna(0.0)
# Fill numeric NaNs with 0.0 so later math and models can run.


## Q1 — Mode of `industry`

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology`
- `healthcare`
- `retail`


In [10]:
q1_mode = df['industry'].value_counts().index[0]
# value_counts sorts by frequency; index[0] is the mode of industry.

print(f"Q1 mode: {q1_mode}")
# Print the mode so it can be matched to the multiple-choice options.


Q1 mode: technology


## Q2 — Correlation matrix (numerical features)

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above.


In [12]:
numerical_df = df.select_dtypes(include=['number'])
# Correlation is defined for numeric columns only.

corr_matrix = numerical_df.corr()
# Pearson correlation matrix over the numeric features.

print("1. interaction_count & lead_score:", corr_matrix.loc['interaction_count', 'lead_score'])
print("2. number_of_courses_viewed & lead_score:", corr_matrix.loc['number_of_courses_viewed', 'lead_score'])
print("3. number_of_courses_viewed & interaction_count:", corr_matrix.loc['number_of_courses_viewed', 'interaction_count'])
print("4. annual_income & interaction_count:", corr_matrix.loc['annual_income', 'interaction_count'])
# Pull the four homework pairs from the matrix and pick the largest.


1. interaction_count & lead_score: 0.9287123432397134
2. number_of_courses_viewed & lead_score: 0.769352062184664
3. number_of_courses_viewed & interaction_count: 0.7216090038937035
4. annual_income & interaction_count: 0.10637458469219016


## Split the data

Use these exact calls. `converted` must not stay in the feature dataframes.


In [15]:
target_col = "converted"

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
# First split: 80% full-train, 20% test. random_state=42 is required.

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
# Second split: 25% of full-train -> 60/20/20 train/val/test overall.

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
# Reset shuffled indexes so later row alignment stays simple.

y_train = df_train[target_col].values
y_val = df_val[target_col].values
y_test = df_test[target_col].values
# Extract the target as numpy arrays.

df_train_features = df_train.drop(columns=[target_col])
df_val_features = df_val.drop(columns=[target_col])
df_test_features = df_test.drop(columns=[target_col])
# Drop converted from the feature frames to avoid leakage.


## Q3 — Mutual information (training set)

Calculate the mutual information score between `converted` and other categorical variables.
Use the training set only. Round with `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source`
- `employment_status`


In [25]:
cat_features = ["industry", "location", "lead_source", "employment_status"]
# The four categorical variables listed in the homework options.

mi_frame = df_train_features[cat_features].fillna("NA")
# Restrict to those columns on train and fill any leftover NaNs.

def calculate_mi(series):
    # Mutual information between one categorical series and y_train.
    return mutual_info_score(series, y_train)

mi_scores = mi_frame.apply(calculate_mi)
# Score each of the four columns. Training set only.

mi_scores = mi_scores.round(2)
# Round to 2 decimals as required.

print("--- Q3: mutual information scores (training set) ---")
print(mi_scores.sort_values(ascending=False))
# Sort descending so the largest score is obvious.


--- Q3: mutual information scores (training set) ---
lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64


## Q4 — Logistic regression accuracy

Include categorical variables with one-hot encoding.
Fit with:

`LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`

Accuracy on validation, rounded to 2 decimal digits.

- 0.55
- 0.65
- 0.75
- 0.85


In [28]:
from sklearn.feature_extraction import DictVectorizer
# One-hot encode categoricals (and pass numerics through) via dicts.

def fill_features(frame):
    # Copy, then fill categorical NaNs with NA and numeric NaNs with 0.0.
    out = frame.copy()
    cat_cols = out.select_dtypes(include=["object", "string"]).columns
    num_cols = out.select_dtypes(include="number").columns
    out[cat_cols] = out[cat_cols].fillna("NA")
    out[num_cols] = out[num_cols].fillna(0.0)
    return out

train_f = fill_features(df_train_features)
val_f = fill_features(df_val_features)
# Fill train and validation before vectorizing.

train_dicts = train_f.to_dict(orient='records')
val_dicts = val_f.to_dict(orient='records')
# DictVectorizer expects a list of row dicts.

dv = DictVectorizer(sparse=False)
# Dense numpy array output.

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
# Fit encoding on train only; transform val with the same mapping.

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# Homework-specified logistic regression.

model.fit(X_train, y_train)
# Fit on the training features and labels.

y_val_pred = model.predict(X_val)
# Class predictions on validation.

val_accuracy = accuracy_score(y_val, y_val_pred)
# Unrounded validation accuracy (also used in Q5).

print(f"Q4 validation accuracy: {round(val_accuracy, 2)}")
# Round to 2 decimals for the form.


Q4 validation accuracy: 0.65


## Q5 — Feature elimination

Train with the same features/parameters as Q4 (without rounding).
Exclude each feature, retrain, record accuracy, compute difference vs original.

Which of the following has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

The difference does not have to be positive.


In [30]:
features_to_test = ["lead_source", "number_of_courses_viewed", "interaction_count"]
# Drop each of these features in turn.

baseline_acc = val_accuracy
# Keep the unrounded Q4 accuracy as the baseline.

print(f"Baseline accuracy (unrounded): {baseline_acc}")

def drop_and_fill(frame, col_to_drop):
    # Drop one column, copy, then fill remaining NaNs.
    out = frame.drop(columns=[col_to_drop], errors='ignore').copy()
    cat_cols = out.select_dtypes(include=["object", "string"]).columns
    num_cols = out.select_dtypes(include="number").columns
    out[cat_cols] = out[cat_cols].fillna("NA")
    out[num_cols] = out[num_cols].fillna(0.0)
    return out


Baseline accuracy (unrounded): 0.645


In [31]:
for col in features_to_test:
    # Retrain without each candidate feature.

    t_f = drop_and_fill(df_train_features, col)
    v_f = drop_and_fill(df_val_features, col)

    t_dicts = t_f.to_dict(orient="records")
    v_dicts = v_f.to_dict(orient="records")

    dv_q5 = DictVectorizer(sparse=False)
    X_t = dv_q5.fit_transform(t_dicts)
    X_v = dv_q5.transform(v_dicts)

    m_q5 = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
    m_q5.fit(X_t, y_train)

    pred_q5 = m_q5.predict(X_v)
    acc_q5 = accuracy_score(y_val, pred_q5)

    diff = baseline_acc - acc_q5
    # Signed difference: original accuracy minus accuracy without the feature.

    print(f"[without {col}] accuracy: {acc_q5:.5f} | difference (baseline - without): {diff:.5f}")


[without lead_source] accuracy: 0.64200 | difference (baseline - without): 0.00300
[without number_of_courses_viewed] accuracy: 0.64300 | difference (baseline - without): 0.00200
[without interaction_count] accuracy: 0.60100 | difference (baseline - without): 0.04400


## Q6 — Regularized logistic regression

Try `C` in `[0.000001, 0.00001, 0.0001, 0.001]`.
Accuracy on validation, rounded to 3 decimal digits.
If multiple options, select the smallest `C`.

- 0.000001
- 0.00001
- 0.0001
- 0.001


In [32]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]
# Official C candidates. Train on the same full Q4 feature set.

for c_val in C_values:
    m = LogisticRegression(solver="liblinear", C=c_val, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_val, m.predict(X_val))
    print(f"C={c_val}  accuracy={acc:.5f}  round3={round(acc, 3)}")
    # Report unrounded and 3-decimal accuracy; pick the best C (smallest C on ties).


C=1e-06  accuracy=0.59800  round3=0.598
C=1e-05  accuracy=0.59800  round3=0.598
C=0.0001  accuracy=0.61300  round3=0.613
C=0.001  accuracy=0.64500  round3=0.645
